In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

In [3]:
# from fairlearn.datasets import fetch_acs_income
# df = fetch_acs_income(cache=True, as_frame=True, return_X_y=False)
# df_full = df.frame
# df_full

In [4]:
def scale_num_features(df: pd.DataFrame, num_features: List[str]):
    scaler = StandardScaler()
    df[num_features] = scaler.fit_transform(df[num_features].values)
    return df

def getACSIncomeData():
    df = pd.read_pickle("../datasets/acs_income.pkl")
    cat_features = ['COW', 'SCHL', 'MAR', 'RELP', 'SEX', 'RAC1P']
    num_features = ['AGEP', 'WKHP']

    df = df.sample(frac=1, random_state=0)
    
    df.drop(columns=['OCCP', 'POBP'], inplace=True)

    df["SCHL_new"] = df["SCHL"].apply(lambda x: 1 if x <= 15 else (x - 14))
    df["SCHL"] = df["SCHL_new"]
    df.drop(columns=["SCHL_new"], inplace=True)

    df = pd.get_dummies(df, columns=cat_features, dtype=float)
    df = scale_num_features(df, num_features=num_features)
    df['PINCP'] = df['PINCP'].apply(lambda x: 1 if x >= 50000 else 0)

    y = df['PINCP']
    X = df.drop(columns=['PINCP'])

    return X, y

In [24]:
df_full = pd.read_pickle("../datasets/acs_income.pkl").sample(frac=1, random_state=0)
df_full

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P,PINCP
1645316,40.0,1.0,21.0,3.0,5740.0,55.0,0.0,40.0,2.0,1.0,40700.0
924583,37.0,1.0,19.0,1.0,8740.0,128.0,0.0,40.0,2.0,1.0,38000.0
682734,59.0,4.0,24.0,3.0,2205.0,36.0,0.0,40.0,1.0,2.0,197000.0
790940,28.0,1.0,21.0,5.0,1460.0,26.0,0.0,40.0,1.0,1.0,90000.0
68445,44.0,2.0,21.0,1.0,3255.0,6.0,0.0,40.0,2.0,1.0,38000.0
...,...,...,...,...,...,...,...,...,...,...,...
152315,17.0,1.0,16.0,5.0,4720.0,6.0,2.0,12.0,2.0,1.0,500.0
963395,55.0,4.0,16.0,1.0,5720.0,8.0,1.0,24.0,2.0,1.0,36000.0
117952,72.0,1.0,19.0,3.0,5400.0,6.0,0.0,40.0,2.0,1.0,48700.0
1484405,55.0,1.0,16.0,3.0,6200.0,35.0,2.0,50.0,1.0,1.0,80000.0


In [17]:
df_full.describe()

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P,PINCP
count,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06
mean,4.341127e+01,2.077500e+00,1.861814e+01,2.521997e+00,4.180517e+03,6.581708e+01,2.241254e+00,3.833390e+01,1.479282e+00,1.874745e+00,5.666386e+04
std,1.530203e+01,1.825338e+00,3.297826e+00,1.796720e+00,2.658717e+03,9.306245e+01,4.385288e+00,1.308073e+01,4.995707e-01,2.084384e+00,7.306745e+04
min,1.700000e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+01,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.040000e+02
25%,3.000000e+01,1.000000e+00,1.600000e+01,1.000000e+00,2.205000e+03,1.800000e+01,0.000000e+00,3.500000e+01,1.000000e+00,1.000000e+00,2.000000e+04
50%,4.300000e+01,1.000000e+00,1.900000e+01,1.000000e+00,4.200000e+03,3.600000e+01,1.000000e+00,4.000000e+01,1.000000e+00,1.000000e+00,3.900000e+04
75%,5.600000e+01,3.000000e+00,2.100000e+01,5.000000e+00,5.740000e+03,4.800000e+01,2.000000e+00,4.400000e+01,2.000000e+00,1.000000e+00,6.800000e+04
max,9.600000e+01,8.000000e+00,2.400000e+01,5.000000e+00,9.830000e+03,5.540000e+02,1.700000e+01,9.900000e+01,2.000000e+00,9.000000e+00,1.423000e+06


In [46]:
df_full.drop(columns=['OCCP', 'POBP'], inplace=True)

In [47]:
df_full

,AGEP,COW,SCHL,MAR,RELP,WKHP,SEX,RAC1P,PINCP
0,18.0,1.0,18.0,5.0,17.0,21.0,2.0,2.0,1600.0
1,53.0,5.0,17.0,5.0,16.0,40.0,1.0,1.0,10000.0
2,41.0,1.0,16.0,5.0,17.0,40.0,1.0,1.0,24000.0
3,18.0,6.0,18.0,5.0,17.0,2.0,2.0,1.0,180.0
4,21.0,5.0,19.0,5.0,17.0,50.0,1.0,1.0,29000.0
...,...,...,...,...,...,...,...,...,...
1664495,39.0,6.0,16.0,5.0,0.0,20.0,1.0,1.0,9600.0
1664496,38.0,6.0,14.0,5.0,0.0,32.0,1.0,8.0,2400.0
1664497,37.0,1.0,19.0,3.0,13.0,40.0,2.0,9.0,19700.0
1664498,47.0,1.0,16.0,1.0,1.0,40.0,1.0,8.0,18700.0


In [ ]:
df_full["SCHL_new"] = df_full["SCHL"].apply(lambda x: 1 if x <= 15 else (x - 14))
df_full["SCHL"] = df_full["SCHL_new"]
df_full.drop(columns=["SCHL_new"], inplace=True)

In [50]:
df_full.describe()

,AGEP,COW,SCHL,MAR,RELP,WKHP,SEX,RAC1P,PINCP
count,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06,1.664500e+06
mean,4.341127e+01,2.077500e+00,4.914338e+00,2.521997e+00,2.241254e+00,3.833390e+01,1.479282e+00,1.874745e+00,5.666386e+04
std,1.530203e+01,1.825338e+00,2.456141e+00,1.796720e+00,4.385288e+00,1.308073e+01,4.995707e-01,2.084384e+00,7.306745e+04
min,1.700000e+01,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.040000e+02
25%,3.000000e+01,1.000000e+00,2.000000e+00,1.000000e+00,0.000000e+00,3.500000e+01,1.000000e+00,1.000000e+00,2.000000e+04
50%,4.300000e+01,1.000000e+00,5.000000e+00,1.000000e+00,1.000000e+00,4.000000e+01,1.000000e+00,1.000000e+00,3.900000e+04
75%,5.600000e+01,3.000000e+00,7.000000e+00,5.000000e+00,2.000000e+00,4.400000e+01,2.000000e+00,1.000000e+00,6.800000e+04
max,9.600000e+01,8.000000e+00,1.000000e+01,5.000000e+00,1.700000e+01,9.900000e+01,2.000000e+00,9.000000e+00,1.423000e+06


In [ ]:
cat_features = ['COW', 'SCHL', 'MAR', 'RELP', 'SEX', 'RAC1P']
num_features = ['AGEP', 'WKHP']

In [52]:
df_full = pd.get_dummies(df_full, columns=cat_features, dtype=float)
df_full = scale_num_features(df_full, num_features=num_features)
df_full['PINCP'] = df_full['PINCP'].apply(lambda x: 1 if x >= 50000 else 0)

In [53]:
df_full

,AGEP,WKHP,PINCP,COW_1.0,COW_2.0,COW_3.0,COW_4.0,COW_5.0,COW_6.0,COW_7.0,...,SEX_2.0,RAC1P_1.0,RAC1P_2.0,RAC1P_3.0,RAC1P_4.0,RAC1P_5.0,RAC1P_6.0,RAC1P_7.0,RAC1P_8.0,RAC1P_9.0
0,-1.660647,-1.325148,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.626631,0.127371,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.157579,0.127371,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1.660647,-2.777666,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.464595,0.891854,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1664495,-0.288280,-1.401596,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1664496,-0.353631,-0.484216,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1664497,-0.418982,0.127371,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1664498,0.234526,0.127371,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [63]:
y = df_full['PINCP']
X = df_full.drop(columns=['PINCP'])

In [80]:
y

0          0
1          0
2          0
3          0
4          0
          ..
1664495    0
1664496    0
1664497    0
1664498    0
1664499    0
Name: PINCP, Length: 1664500, dtype: int64

In [70]:
X, y = getACSIncomeData()

In [71]:
X

,AGEP,WKHP,COW_1.0,COW_2.0,COW_3.0,COW_4.0,COW_5.0,COW_6.0,COW_7.0,COW_8.0,...,SEX_2.0,RAC1P_1.0,RAC1P_2.0,RAC1P_3.0,RAC1P_4.0,RAC1P_5.0,RAC1P_6.0,RAC1P_7.0,RAC1P_8.0,RAC1P_9.0
0,-1.660647,-1.325148,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.626631,0.127371,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.157579,0.127371,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1.660647,-2.777666,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.464595,0.891854,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1664495,-0.288280,-1.401596,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1664496,-0.353631,-0.484216,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1664497,-0.418982,0.127371,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1664498,0.234526,0.127371,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [56]:
data_set = IncomeDataset()
train_data, test_data = data_set.get_data(0)
X_train, y_train = train_data
X_test, y_test = test_data

In [54]:
base_model = LR()
base_model.train(X_train.values, y_train.values)

In [55]:
np.sum(np.abs(base_model.predict(X_train.values) - y_train.values)) / len(y_train)

np.float64(0.2271613097026134)

In [52]:
np.sum(np.abs(base_model.predict(X_test.values) - y_test.values)) / len(y_test)

np.float64(0.22728266746770803)

### NN

In [8]:
data_set = IncomeDataset()
train_data, test_data = data_set.get_data(0)
X_train, y_train = train_data
X_test, y_test = test_data

base_model = NN(X_train.shape[1])
base_model.train(X_train.values, y_train.values, verbose=1)

Epoch 0: train loss: 0.6906492710113525
Epoch 1: train loss: 0.6862571835517883
Epoch 2: train loss: 0.6821297407150269
Epoch 3: train loss: 0.6781940460205078
Epoch 4: train loss: 0.6743854284286499
Epoch 5: train loss: 0.6706457734107971
Epoch 6: train loss: 0.6669087409973145
Epoch 7: train loss: 0.6630986928939819
Epoch 8: train loss: 0.6591463088989258
Epoch 9: train loss: 0.6549786925315857
Epoch 10: train loss: 0.6505325436592102
Epoch 11: train loss: 0.6457536816596985
Epoch 12: train loss: 0.6405906677246094
Epoch 13: train loss: 0.634998619556427
Epoch 14: train loss: 0.6289404630661011
Epoch 15: train loss: 0.6223869323730469
Epoch 16: train loss: 0.6153113842010498
Epoch 17: train loss: 0.6077033877372742
Epoch 18: train loss: 0.599578320980072
Epoch 19: train loss: 0.5909833312034607
Epoch 20: train loss: 0.5820037722587585
Epoch 21: train loss: 0.572772204875946
Epoch 22: train loss: 0.5634620785713196
Epoch 23: train loss: 0.5542765259742737
Epoch 24: train loss: 0.54541

In [9]:
np.sum(np.abs(base_model.predict(X_train.values) - y_train.values)) / len(y_train)

np.float64(0.21944848302793632)

In [10]:
np.sum(np.abs(base_model.predict(X_test.values) - y_test.values)) / len(y_test)

np.float64(0.22087834184439772)

In [ ]:
# torch.save(base_model.model.state_dict(), "../results/recourse_model/income_0.pth")

In [15]:
data_set = IncomeDataset()
train_data, test_data = data_set.get_data(1)
X_train, y_train = train_data
X_test, y_test = test_data

base_model = NN(X_train.shape[1])
base_model.train(X_train.values, y_train.values, verbose=1)

Epoch 0: train loss: 0.6906533241271973
Epoch 1: train loss: 0.6862615942955017
Epoch 2: train loss: 0.6821350455284119
Epoch 3: train loss: 0.678200364112854
Epoch 4: train loss: 0.6743934154510498
Epoch 5: train loss: 0.6706556677818298
Epoch 6: train loss: 0.6669219136238098
Epoch 7: train loss: 0.6631178855895996
Epoch 8: train loss: 0.6591777801513672
Epoch 9: train loss: 0.6550331115722656
Epoch 10: train loss: 0.650622546672821
Epoch 11: train loss: 0.6458896994590759
Epoch 12: train loss: 0.6407774686813354
Epoch 13: train loss: 0.6352354884147644
Epoch 14: train loss: 0.6292197108268738
Epoch 15: train loss: 0.6226962804794312
Epoch 16: train loss: 0.6156388521194458
Epoch 17: train loss: 0.6080406904220581
Epoch 18: train loss: 0.5999251008033752
Epoch 19: train loss: 0.5913422107696533
Epoch 20: train loss: 0.5823779106140137
Epoch 21: train loss: 0.5731605291366577
Epoch 22: train loss: 0.5638589262962341
Epoch 23: train loss: 0.5546765327453613
Epoch 24: train loss: 0.5458

In [16]:
np.sum(np.abs(base_model.predict(X_train.values) - y_train.values)) / len(y_train)

np.float64(0.2207990387503755)

In [17]:
np.sum(np.abs(base_model.predict(X_test.values) - y_test.values)) / len(y_test)

np.float64(0.2197644938419946)

In [ ]:
# torch.save(base_model.model.state_dict(), "../results/recourse_model/income_1.pth")

### Folktables

In [11]:
data_set = SBADataset()
train_data, test_data = data_set.get_data(1)
X_train, y_train = train_data
X_test, y_test = test_data

In [13]:
X_test

,Zip,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,...,GrAppv,SBA_Appv,Portion,daysterm,xx,RevLineCr_-1,RevLineCr_0,RevLineCr_N,RevLineCr_T,RevLineCr_Y
1609,1.027917,-1.012018,0.468049,0.354715,0.904285,-0.203553,-0.376699,0.147856,-0.265297,-0.192770,...,1.957623,2.479424,1.631627,0.904285,1.036093,0.0,0.0,1.0,0.0,0.0
1197,1.393135,-1.176647,-0.174351,-0.159500,1.502553,0.853652,-0.376699,-0.313113,-0.265297,-0.192770,...,-0.388851,-0.395990,0.232805,1.502553,1.301532,0.0,0.0,1.0,0.0,0.0
1654,-0.596500,-1.012018,0.543171,0.611822,-0.651213,-0.111622,-0.376699,-0.082628,0.068014,-0.192770,...,-0.723217,-0.716059,-1.166017,-0.651213,-0.378053,0.0,0.0,0.0,0.0,1.0
1288,1.008579,0.853786,-2.351492,-2.473464,-0.292252,-0.088639,2.613674,-0.313113,-0.265297,6.772448,...,-0.316948,-0.274711,0.792334,-0.292252,-1.218913,0.0,0.0,1.0,0.0,0.0
1677,-1.258975,1.184875,0.571254,0.611822,-0.651213,-0.249518,-0.376699,-0.313113,-0.181969,-0.192770,...,-0.782396,-0.750754,-1.166017,-0.651213,-0.359747,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1715,-0.515832,0.817202,0.600742,0.611822,-0.292252,-0.226535,-0.376699,-0.313113,-0.265297,-0.192770,...,-0.590062,-0.546917,0.792334,-0.292252,-0.011626,0.0,0.0,1.0,0.0,0.0
411,-0.745129,0.636109,-0.871514,-0.930821,1.502553,-0.180570,-0.376699,-0.313113,-0.265297,-0.192770,...,-0.279369,-0.299709,0.232805,1.502553,0.985751,0.0,1.0,0.0,0.0,0.0
1927,1.644533,0.853786,-1.556741,-1.445035,-0.830693,-0.203553,2.613674,-0.313113,-0.265297,-0.192770,...,-0.702504,-0.665403,0.512569,-0.830693,-1.434315,0.0,0.0,1.0,0.0,0.0
1833,1.345066,0.853786,-1.805276,-1.702142,-1.010174,-0.226535,-0.376699,-0.313113,-0.265297,-0.192770,...,-0.678832,-0.627584,1.072098,-1.010174,-1.683278,0.0,0.0,1.0,0.0,0.0


In [3]:
from folktables import ACSDataSource, ACSIncome

In [ ]:
datasource = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
ca_data = datasource.get_data(states=['CA'], download=False)

In [12]:
ca_features, ca_labels, _ = ACSIncome.df_to_pandas(ca_data)